# Plot training 

In [5]:
using Plots
using Statistics

# parse the log
lines = readlines("training_log.txt")
epoch_lines = filter(l -> startswith(l, "Epoch"), lines)

epochs       = Int[]
train_losses = Float32[]
train_accs   = Float32[]
val_accs     = Float32[]
sparsities1  = Float32[]

for line in epoch_lines
    e    = parse(Int,     match(r"Epoch (\d+)", line)[1])
    tl   = parse(Float32, match(r"train_loss=([0-9.e+]+)", line)[1])
    ta   = parse(Float32, match(r"train_acc=([0-9.]+)", line)[1])
    va   = parse(Float32, match(r"val_acc=([0-9.]+)", line)[1])
    sp1  = parse(Float32, match(r"sparsity_dense1=([0-9.]+)", line)[1])

    push!(epochs, e)
    push!(train_losses, tl)
    push!(train_accs, ta)
    push!(val_accs, va)
    push!(sparsities1, sp1)
end

# smooth helper
function smooth(x; window=5)
    n = length(x)
    [mean(x[max(1,i-window÷2):min(n,i+window÷2)]) for i in 1:n]
end

p1 = scatter(epochs, train_losses, label="train loss",
             xlabel="Epoch", ylabel="Loss", title="Loss",
             markersize=4, alpha=0.5, color=:blue)
     plot!(epochs, smooth(train_losses), label="", 
           color=:blue, linewidth=2)

p2 = scatter(epochs, train_accs, label="train acc",
             xlabel="Epoch", ylabel="Accuracy", title="Accuracy",
             markersize=4, alpha=0.5, color=:blue)
     plot!(epochs, smooth(train_accs), label="",
           color=:blue, linewidth=2)
     scatter!(epochs, val_accs, label="val acc",
              markersize=4, alpha=0.5, color=:red)
     plot!(epochs, smooth(val_accs), label="",
           color=:red, linewidth=2)

p3 = scatter(epochs, sparsities1, label="sparsity dense1",
             xlabel="Epoch", ylabel="Sparsity", title="MLP Sparsity",
             markersize=4, alpha=0.5, color=:green)
     plot!(epochs, smooth(sparsities1), label="",
           color=:green, linewidth=2)

plot(p1, p2, p3, layout=(3,1), size=(800, 900))
savefig("training_curves.png")

"/Users/andreibleahu/Documents/BayesGPT-I/training_curves.png"

# Aim : Test inference of model

In [2]:
import Pkg

In [3]:
Pkg.activate(@__DIR__)

  Activating project at `~/Documents/BayesGPT-I`


In [7]:
using Flux, BSON, Statistics, Printf
include("src/BayesGPT.jl")
include("generate_only.jl")

  Activating 

Tokenizer loaded ✓ — vocab size: 16330


project at `~/Documents/BayesGPT-I`


Model loaded ✓

Seed: "the ancient horror"
──────────────────────────────────────────────────
Sample 1: the ancient horror caught caught caught
Sample 2: the ancient horror caught caught caught
Sample 3: the ancient horror horror caught caught

[Top-k=50, penalty=1.5] Seed: "the ancient horror"
──────────────────────────────────────────────────
Sample 1: the ancient horror in foot going
Sample 2: the ancient horror caught stained going
Sample 3: the ancient horror as going in

[Bayesian n_forward=20, penalty=1.5] Seed: "the ancient horror"
──────────────────────────────────────────────────
Generated : the ancient horror in going begged caught caught

Token uncertainty (σ) and 95% CI:
──────────────────────────────────────────────────
  in              σ=0.529  CI=±1.036  █████████████████████ uncertain
  going           σ=0.520  CI=±1.018  █████████████████████ uncertain
  begged          σ=0.359  CI=±0.704  ██████████████ uncertain
  caught          σ=0.220  CI=±0.431  █████████ moder

In [8]:
BSON.@load "tokenizer.bson" tokenizer
BSON.@load "model.bson" model

## Inspect model

In [9]:
model

Chain(
  BSON.__deserialized_types__.var"##239"(),
  Embed(64 × 16330),                    # 1_045_120 parameters
  PositionEncoding(dim=64, max_len=1000),
  Dropout(0.1),
  TransformerDecoderBlock(
    CausalMultiheadAttention(
      Dense(64 => 64; bias=false),      # 4_096 parameters
      Dense(64 => 64; bias=false),      # 4_096 parameters
      Dense(64 => 64; bias=false),      # 4_096 parameters
      Dense(64 => 64),                  # 4_160 parameters
    ),
    LayerNorm(64),                      # 128 parameters
    VariationalDropout(64→256, relu),   # 33_024 parameters
    VariationalDropout(256→64, identity),  # 32_832 parameters
    LayerNorm(64),                      # 128 parameters
    Dropout(0.1),
  ),
  TransformerDecoderBlock(
    CausalMultiheadAttention(
      Dense(64 => 64; bias=false),      # 4_096 parameters
      Dense(64 => 64; bias=false),      # 4_096 parameters
      Dense(64 => 64; bias=false),      # 4_096 parameters
      Dense(64 => 64),            

In [13]:
for (i, layer) in enumerate(model.layers)
    println("Layer $i: ", typeof(layer))
end

Layer 1: BSON.__deserialized_types__.var"##239"
Layer 2: Embed{Matrix{Float32}}
Layer 3: PositionEncoding{Matrix{Float32}}
Layer 4: Dropout{Float32, Colon, TaskLocalRNG}
Layer 5: TransformerDecoderBlock{CausalMultiheadAttention{Dense{typeof(identity), Matrix{Float32}, Bool}, Dense{typeof(identity), Matrix{Float32}, Bool}, Dense{typeof(identity), Matrix{Float32}, Bool}, Dense{typeof(identity), Matrix{Float32}, Vector{Float32}}}, LayerNorm{typeof(identity), Flux.Scale{typeof(identity), Vector{Float32}, Vector{Float32}}, Float32, 1}, VariationalDropoutMolchanov{typeof(relu), Matrix{Float32}, Vector{Float32}}, VariationalDropoutMolchanov{typeof(identity), Matrix{Float32}, Vector{Float32}}, LayerNorm{typeof(identity), Flux.Scale{typeof(identity), Vector{Float32}, Vector{Float32}}, Float32, 1}, Dropout{Float64, Colon, TaskLocalRNG}}
Layer 6: TransformerDecoderBlock{CausalMultiheadAttention{Dense{typeof(identity), Matrix{Float32}, Bool}, Dense{typeof(identity), Matrix{Float32}, Bool}, Dense{t

In [14]:
tokenizer

IndexTokenizer(vocab_size=16330, unk="<UNK>")

So, we need to use the tokenizer to tokenize the seed before we move forward.

In [22]:
function _tokenize_seed(tokenizer, seed::AbstractString)
    words     = [m.match for m in eachmatch(r"\w+\b", simplify(seed))]
    token_ids = tokenizer(words)
    if isempty(token_ids)
        @warn "Seed produced no known tokens, using <UNK>"
        return [tokenizer.unkidx]
    end
    token_ids
end

_tokenize_seed (generic function with 1 method)

Let's test it on a couple of strings

In [93]:
string_1 = "the quantum computer feels"

"the quantum computer feels"

In [24]:
string_2 = "Please explain the nature of life to me. Try to be brief, and direct"

"Please explain the nature of life to me. Try to be brief, and direct"

## Let's see how it works on these couple of examples

In [94]:
generated = _tokenize_seed(tokenizer, string_1)

4-element Vector{Int64}:
     3
     2
 15813
  2837

In [95]:
length(split(string_1))


4

Note:

- Now, they match

In [96]:
x = reshape(generated, 1, length(generated), 1)

1×4×1 Array{Int64, 3}:
[:, :, 1] =
 3  2  15813  2837

In [97]:
logits_all = model(x)

16330×4×1 Array{Float32, 3}:
[:, :, 1] =
  -0.346117   -0.156303   -0.28814   -3.29819
   6.82815     0.282587    3.27647    1.80931
   1.90186     7.21442     6.39845    4.10948
   5.83425     8.1116      7.76261    8.92179
   9.13721     9.94174    11.5839    11.0115
   4.37155     5.26261     5.72779    6.40603
  11.2915     13.2421     14.1755    17.2461
  10.0666     11.7336     10.8034    10.2828
   5.30554     5.6509      6.02057    6.60097
   1.14228     4.21349     3.457      1.90883
   6.70806     7.27815     8.91732   14.8367
   1.93859     4.31496     3.54881   -2.63062
   4.42247     4.12897     5.35171    4.11832
   ⋮                                 
  -3.51774    -4.24328    -4.96877   -5.57396
  -2.53471    -0.962479   -2.02157   -3.22098
  -0.900712    2.55517    -1.79253   -3.66812
  -9.29762    -9.98666   -10.7895    -9.74437
  -0.302376   -2.54912    -3.61007    3.01739
  -7.11075    -9.0539     -8.44653   -6.72794
  -7.63474   -10.1514     -8.44819   -9.23047
   7.

In [98]:
seq_len = size(logits_all, 2)
next_logits = logits_all[:, seq_len, 1]

16330-element Vector{Float32}:
 -3.2981868
  1.8093107
  4.1094775
  8.921791
 11.011529
  6.4060316
 17.246124
 10.28279
  6.6009665
  1.9088292
 14.8367195
 -2.6306236
  4.1183176
  ⋮
 -5.5739555
 -3.220979
 -3.6681235
 -9.744366
  3.0173924
 -6.727941
 -9.230474
  0.59711206
 -4.618858
  3.06835
 -4.29881
 -9.811582

In [99]:
best_id = argmax(next_logits)
println("Greedy next word: ", tokenizer.vocabulary[best_id])

Greedy next word: going


In [100]:
next_id = argmax(next_logits)  # greedy for now

291

In [101]:
println("Predicted: ", tokenizer.vocabulary[next_id])
push!(generated, next_id)
println("New sequence: ", join([tokenizer.vocabulary[id] for id in generated], " "))

Predicted: going
New sequence: the <UNK> computer feels going


In [102]:
generated

5-element Vector{Int64}:
     3
     2
 15813
  2837
   291

In [103]:
x = reshape(generated, 1, length(generated), 1)
next_logits = model(x)[:, end, 1]
next_id = argmax(next_logits)

println("Predicted: ", tokenizer.vocabulary[next_id])
push!(generated, next_id)
println("New sequence: ", join([tokenizer.vocabulary[id] for id in generated], " "))

Predicted: going
New sequence: the <UNK> computer feels going going


In [89]:
get(tokenizer.lookup, "a", nothing)

Note:

- It seems like a clear overfitting problem

In [105]:
total_params = sum(length, Flux.params(model))
println("Total parameters: ", total_params)

┌ Warning: `Flux.params(m...)` is deprecated. Use `Flux.trainable(model)` for parameter collection,
│ and the explicit `gradient(m -> loss(m, x, y), model)` for gradient computation.
└ @ Flux ~/.julia/packages/Flux/9PibT/src/deprecations.jl:93


Total parameters: 2271690


In [106]:
# How many words appear less than 5 times?
text = join([read("data/$f", String) for f in readdir("data") if endswith(f, ".txt")], " ")
word_counts = Dict{String,Int}()
for w in split(text)
    word_counts[w] = get(word_counts, w, 0) + 1
end

rare = sum(v < 5 for v in values(word_counts))
println("Rare words (<5 occurrences): ", rare, " / ", length(word_counts))

Rare words (<5 occurrences): 62343 / 78405


In [107]:
vocab_words = Set(tokenizer.vocabulary)
kept_rare = sum(v < 5 for (w, v) in word_counts if w in vocab_words)
kept_common = sum(v >= 5 for (w, v) in word_counts if w in vocab_words)
println("Kept vocab - rare (<5):   ", kept_rare)
println("Kept vocab - common (≥5): ", kept_common)

# frequency distribution
for threshold in [5, 10, 50, 100, 500]
    n = sum(v >= threshold for (w,v) in word_counts if w in vocab_words)
    println("Words with ≥$threshold occurrences: ", n)
end

Kept vocab - rare (<5):   6322
Kept vocab - common (≥5): 7986
Words with ≥5 occurrences: 7986
Words with ≥10 occurrences: 5048
Words with ≥50 occurrences: 1369
Words with ≥100 occurrences: 684
Words with ≥500 occurrences: 178
